# Part 3: Joins and Aggregations

Most interesting questions require combining data from multiple tables. In this notebook we cover:

- `INNER JOIN` — rows that match in both tables
- `LEFT JOIN` — all rows from the left table, NULLs where no match
- Multi-table joins
- Aggregate functions: `AVG`, `MAX`, `MIN`, `SUM`, `COUNT`
- `HAVING` — filtering on aggregated results

> **Prerequisite:** Run `01_setup.ipynb` first.

In [ ]:
import sqlite3
import pandas as pd

con = sqlite3.connect("gene_expression.db")

def query(sql, params=()):
    return pd.read_sql(sql, con, params=params)

print("Connected.")

---
## 1. INNER JOIN — combine expression with gene and sample info

An `INNER JOIN` returns only rows where the join condition matches in **both** tables.

In [ ]:
query("""
SELECT g.symbol, s.name AS sample, s.condition, s.tissue,
       e.tpm, e.raw_counts
FROM   expression e
JOIN   genes      g ON g.gene_id   = e.gene_id
JOIN   samples    s ON s.sample_id = e.sample_id
ORDER BY g.symbol, s.name
LIMIT 20
""")

---
## 2. LEFT JOIN — find genes with no expression data

A `LEFT JOIN` returns **all rows from the left table**, even when there is no matching row in the right table (those fields will be `NULL`).

Here we deliberately show the pattern — all genes currently have expression data, so we filter to illustrate the NULL concept.

In [ ]:
query("""
SELECT g.symbol, g.chromosome, e.expr_id
FROM   genes      g
LEFT JOIN expression e ON e.gene_id = g.gene_id
WHERE  e.expr_id IS NULL   -- only genes with NO expression data
""")

*(Empty — every gene has data. Try deleting some rows and re-running to see NULLs appear.)*

---
## 3. Aggregations — statistics per gene

In [ ]:
query("""
SELECT g.symbol,
       COUNT(e.expr_id)          AS n_samples,
       ROUND(AVG(e.tpm), 2)      AS avg_tpm,
       ROUND(MIN(e.tpm), 2)      AS min_tpm,
       ROUND(MAX(e.tpm), 2)      AS max_tpm,
       ROUND(MAX(e.tpm) - MIN(e.tpm), 2) AS range_tpm
FROM   expression e
JOIN   genes      g ON g.gene_id = e.gene_id
GROUP BY g.gene_id
ORDER BY avg_tpm DESC
""")

---
## 4. Aggregations — statistics per sample

In [ ]:
query("""
SELECT s.name, s.condition, s.tissue,
       COUNT(e.expr_id)      AS n_genes,
       ROUND(AVG(e.tpm), 2) AS avg_tpm,
       ROUND(SUM(e.tpm), 2) AS total_tpm
FROM   expression e
JOIN   samples    s ON s.sample_id = e.sample_id
GROUP BY s.sample_id
ORDER BY avg_tpm DESC
""")

---
## 5. HAVING — filter groups after aggregation

`WHERE` filters individual rows *before* grouping. `HAVING` filters *after* grouping.

Find genes with average TPM > 800 across all samples:

In [ ]:
query("""
SELECT g.symbol,
       ROUND(AVG(e.tpm), 2) AS avg_tpm
FROM   expression e
JOIN   genes      g ON g.gene_id = e.gene_id
GROUP BY g.gene_id
HAVING avg_tpm > 800
ORDER BY avg_tpm DESC
""")

---
## 6. Subquery — tumor vs normal expression for each gene

Compute average TPM separately for tumor and normal samples and compare them in one result.

In [ ]:
query("""
SELECT g.symbol,
       ROUND(AVG(CASE WHEN s.condition = 'tumor'  THEN e.tpm END), 2) AS tumor_avg_tpm,
       ROUND(AVG(CASE WHEN s.condition = 'normal' THEN e.tpm END), 2) AS normal_avg_tpm,
       ROUND(
           AVG(CASE WHEN s.condition = 'tumor'  THEN e.tpm END) -
           AVG(CASE WHEN s.condition = 'normal' THEN e.tpm END),
       2) AS delta_tpm
FROM   expression e
JOIN   genes      g ON g.gene_id   = e.gene_id
JOIN   samples    s ON s.sample_id = e.sample_id
GROUP BY g.gene_id
ORDER BY ABS(delta_tpm) DESC
""")

---
## Exercises

**Q1.** Which sample has the highest average TPM across all genes?

In [ ]:
# Your query here


**Q2.** Find all genes where the maximum raw_counts across all samples exceeds 1800.

In [ ]:
# Your query here


**Q3.** For each chromosome, what is the average TPM of all genes on that chromosome? Order by average TPM descending.

In [ ]:
# Your query here


**Q4.** List genes whose expression range (MAX - MIN tpm) is in the top 5 across all samples — these are the most variably expressed genes.

In [ ]:
# Your query here


In [ ]:
con.close()